# Pipeline : Mapping, SNP Calling and SDPOP for *Laurus azorica*

# 1. Mapping and SNP calling

## 1.0 Before running the WORKFLOW

In [ ]:
# Create the directories
mkdir /home/tbertrand/work/L_nobilis_all
mkdir /home/tbertrand/work/L_nobilis_all/GENOME
mkdir /home/tbertrand/work/L_nobilis_all/INPUT
mkdir /home/tbertrand/work/L_nobilis_all/MAPPING
mkdir /home/tbertrand/work/L_nobilis_all/GENOTYPING
mkdir /home/tbertrand/work/L_nobilis_all/WORKFLOW_MAPPING_FREEBAYES

# Download the fasta_generate_regions.py script
cd /home/tbertrand/work/L_nobilis_all/GENOTYPING
wget https://raw.githubusercontent.com/freebayes/freebayes/master/scripts/fasta_generate_regions.py

# Create the input list in the following format (Tab separated)  
R1.fasta    R2.fasta    Indiv_name

# Exemple: 
/home/tbertrand/work/RNA_data/trimming/LN-T1_Mb_R1_001_val_1.fq.gz  /home/tbertrand/work/RNA_data/trimming/LN-T1_Mb_R2_001_val_2.fq.gz	LN-T1_M
/home/tbertrand/work/RNA_data/trimming/LN-T2_F_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T2_F_R2_001_val_2.fq.gz	LN-T2_F
/home/tbertrand/work/RNA_data/trimming/LN-T3_F_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T3_F_R2_001_val_2.fq.gz	LN-T3_F
/home/tbertrand/work/RNA_data/trimming/LN-T3_Female_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T3_Female_R2_001_val_2.fq.gz	LN-T3_F
/home/tbertrand/work/RNA_data/trimming/LN-T4_M_R1_001_val_1.fq.gz	/home/tbertrand/work/RNA_data/trimming/LN-T4_M_R2_001_val_2.fq.gz	LN-T4_M

# Save it in the INPUT directory
"/home/tbertrand/work/L_novocanariensis/INPUT/reads_list_novocanariensis.txt"

##  1.1 Indexing the reference genome with STAR

In [ ]:
#!/bin/bash
#SBATCH --job-name=index
#SBATCH -p unlimitq
#SBATCH --time=7-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=24

#############################
# 1. MODULES
#############################

module load bioinfo/STAR/2.7.11b

#############################
# 2. FILES 
#############################

WORKDIR=/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOME/Index
FASTA=/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOME/GCA_977007225.1_dmLauAzor1.pri_genomic.fna
ANNOTATION=/home/tbertrand/work/L_azorica/L_azorica_L_azorica/GENOME/braker.fixed_ids.gff3

#############################
# 3. RUN STAR genomeGenerate
#############################

STAR --runMode genomeGenerate --genomeDir ${WORKDIR}  --genomeFastaFiles ${FASTA}  --runThreadN 24 --sjdbGTFfile ${ANNOTATION}  --sjdbGTFtagExonParentTranscript Parent

## 1.2 Snakemake pipeline

#### config.yaml

In [ ]:
genome_dir: "/home/tbertrand/work/L_nobilis_all/GENOME/Index"
ref_genome: "/home/tbertrand/work/L_nobilis_all/GENOME/Index/GCA_977007225.1_dmLauAzor1.pri_genomic.fna"
sample_table: "/home/tbertrand/work/L_nobilis_all/INPUT/reads_list_L_nobilis_all.txt"
mapping_dir: "/home/tbertrand/work/L_nobilis_all/MAPPING"
genotyping_dir: "/home/tbertrand/work/L_nobilis_all/GENOTYPING"

#### snakefile freebayes

In [ ]:
configfile: "config.yaml"

# A METTRE APRES AVOIR FAIT wc -l genotyping_dir/regions/all_regions.bed
NREGIONS = 13338
regions = list(range(1, NREGIONS + 1))

# Lire table samples
samples = {}
with open(config["sample_table"]) as f:
    for line in f:
        r1, r2, ind = line.strip().split()
        samples[ind] = {"r1": r1, "r2": r2}

INDIVIDUALS = list(samples.keys())


rule all:
    input:
        # tous les BAM traités pour freebayes
        expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam", ind=INDIVIDUALS),
        # le fichier BAM_LIST.txt pour freebayes 
        config["genotyping_dir"] + "/BAM_LIST.txt",
        # Tous les VCF chunks générés (pour que Snakemake relance les manquants)
        expand(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf", i=regions),
        # VCF final concaténé
        config["genotyping_dir"] + "/L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
        

#################################
# STAR mapping
#################################

rule star_map:
    input:
        r1=lambda wc: samples[wc.ind]["r1"],
        r2=lambda wc: samples[wc.ind]["r2"]
    output:
        coord=config["mapping_dir"] + "/{ind}_Aligned.sortedByCoord.out.bam",
        transcriptome=config["mapping_dir"] + "/{ind}_Aligned.toTranscriptome.out.bam"
    threads: 8
    resources:
        mem_mb=96000,
        star_slots=1,
        runtime = 2000
    shell:
        """
        module load bioinfo/STAR/2.7.11b

        STAR \
          --genomeDir {config[genome_dir]} \
          --readFilesIn {input.r1} {input.r2} \
          --readFilesCommand gunzip -c \
          --runThreadN {threads} \
          --quantMode TranscriptomeSAM \
          --outFileNamePrefix {config[mapping_dir]}/{wildcards.ind}_ \
          --outSAMtype BAM SortedByCoordinate
        """


#################################
# BAM processing
#################################

rule process_bam:
    input:
        coord=config["mapping_dir"] + "/{ind}_Aligned.sortedByCoord.out.bam",
        transcriptome=config["mapping_dir"] + "/{ind}_Aligned.toTranscriptome.out.bam"
    output:
        final=config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam"
    threads: 4
    resources:
        mem_mb=32000,
        process_bam_slots=1,
        runtime = 1000
    shell:
        """
        module load bioinfo/samtools/1.23

        samtools sort -@ {threads} {input.transcriptome} \
            -o {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam

        samtools flagstat {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam \
            > {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.flagstat

        samtools view {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam | cut -f1 \
            > {config[mapping_dir]}/{wildcards.ind}_transcriptome_reads.txt

        samtools view -N {config[mapping_dir]}/{wildcards.ind}_transcriptome_reads.txt -bh {input.coord} \
            > {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.bam

        samtools addreplacerg \
            -r ID:{wildcards.ind} \
            -r SM:{wildcards.ind} \
            -r PL:ILLUMINA \
            -r LB:{wildcards.ind} \
            -r PU:{wildcards.ind} \
            -o {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.rg.bam \
            {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.bam

        samtools sort -@ {threads} \
            -o {output.final} \
            {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.rg.bam

        samtools index {output.final}

        rm -f {input.coord}
        rm -f {input.transcriptome}
        rm -f {config[mapping_dir]}/{wildcards.ind}_Aligned.toTranscriptome.sorted.out.bam
        rm -f {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.bam
        rm -f {config[mapping_dir]}/{wildcards.ind}_TranscriptomeGenomePos.rg.bam
        rm -f {config[mapping_dir]}/{wildcards.ind}_transcriptome_reads.txt
        """

##################################
# règle pour générer BAM_LIST.txt
##################################

rule make_bam_list:
    input:
        expand(config["mapping_dir"] + "/{ind}_TranscriptomeGenomePos.sorted.rg.bam", ind=INDIVIDUALS)
    output:
        config["genotyping_dir"] + "/BAM_LIST.txt"
    run:
        with open(output[0], "w") as out:
            for bam in input:
                out.write(bam + "\n")

###########################################
# RULE : Generate Freebayes regions
###########################################

rule GenerateFreebayesRegions:
    input:
        index=config["ref_genome"] + ".fai"
    output:
        expand(config["genotyping_dir"] + "/regions/region.{i}.bed", i=regions)
    shell:
        """
        module load bioinfo/freebayes/1.3.6
        module load devel/Miniforge/Miniforge3
        module load bioinfo/vcflib/1.0.12
        mkdir -p {config[genotyping_dir]}/regions

        python {config[genotyping_dir]}/fasta_generate_regions.py \
        {input.index} 100000 \
        > {config[genotyping_dir]}/regions/all_regions.bed

        awk '{{ 
            gsub(":", "\\t"); 
            gsub("-", "\\t"); 
            print > "{config[genotyping_dir]}/regions/region." NR ".bed" 
        }}' {config[genotyping_dir]}/regions/all_regions.bed
        """

###########################################
# RULE : Freebayes avec retries 
###########################################

def get_mem_mb(wildcards, attempt):
    """
    Retourne la mémoire en MB selon le numéro de tentative.
    Attempt=1 → 10 GB
    Attempt=2 → 50 GB
    Attempt=3 → 100 GB
    Attempt=4 → 200 GB

    """
    if attempt == 1:
        return 10000
    elif attempt == 2:
        return 50000
    elif attempt == 3:
        return 100000
    else:
        return 200000

rule VariantCallingFreebayes:
    input:
        ref     = config["ref_genome"],
        regions = config["genotyping_dir"] + "/regions/region.{i}.bed",
        bamlist = config["genotyping_dir"] + "/BAM_LIST.txt"
    output:
        temp(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf")
    log:
        config["genotyping_dir"] + "/logs/freebayes_region.{i}.log"
    threads: 1
    retries: 3
    resources:
        mem_mb = get_mem_mb,
        runtime = 2880
    shell:
        """
	echo "Memory: {resources.mem_mb} MB" >> {log}
        module load bioinfo/freebayes/1.3.6
        module load devel/Miniforge/Miniforge3
        module load bioinfo/vcflib/1.0.12

        mkdir -p {config[genotyping_dir]}/vcf_chunks

        freebayes \
          -f {input.ref} \
          -t {input.regions} \
          -L {input.bamlist} \
          -C 2 \
           -g 1500000 \ 
          --report-monomorphic \
          --use-best-n-alleles 2 \
          > {output} 2>> {log}
        """


###########################################
# RULE : Concatenate VCF
###########################################

rule ConcatVCFs:
    input:
        calls=expand(config["genotyping_dir"] + "/vcf_chunks/variants.{i}.vcf", i=regions)
    output:
        config["genotyping_dir"] + "/L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
    log:
        "logs/ConcatVCFs.log"
    threads: 4
    shell:
        """
        module load bioinfo/Bcftools/1.9

        bcftools concat {input.calls} -Ov -o {output}

#### run_snakefile_freebayes.sh

In [ ]:
#!/bin/bash
#SBATCH --job-name=Snakemake_L_nobilis_all_freebayes
#SBATCH -p unlimitq
#SBATCH --time=20-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --cpus-per-task=1
#SBATCH --mem=8G
#SBATCH --output=snakemake_%j.log

module load bioinfo/Snakemake/8.20.3

snakemake \
  --snakefile snakefile_freebayes \
  --cores 48\
  --rerun-incomplete \
  --keep-going \
  --latency-wait 60 \
  --executor slurm \
  --jobs 12 \
  --resources star_slots=3  --unlock

## 1.3 Results MAPPING

#### Script statistiques mapping: **stats_mapping.sh**

In [ ]:
#!/bin/bash

output="mapping_summary.csv"
echo "Sample,Reads,Unique,Multi,Too_short,Other" > "$output"

for file in *Log.final.out
do
    sample=$(basename "$file" Log.final.out)

    reads=$(grep "Number of input reads" "$file" | awk '{print $NF}')
    unique=$(grep "Uniquely mapped reads %" "$file" | awk '{print $NF}')
    multi=$(grep "% of reads mapped to multiple loci" "$file" | awk '{print $NF}')
    short=$(grep "% of reads unmapped: too short" "$file" | awk '{print $NF}')
    other=$(grep "% of reads unmapped: other" "$file" | awk '{print $NF}')

    echo "$sample,$reads,$unique,$multi,$short,$other" >> "$output"
done

### Results Mapping

In [23]:
import pandas as pd

data = """Sample,Reads,Unique,Multi,Too_short,Other
GRE1-2_5b,58577068,91.08%,4.19%,4.14%,0.38%
GRE3-4_1b,56462041,90.49%,3.96%,4.81%,0.58%
GRE3-4_2b,54599420,92.13%,4.17%,3.19%,0.32%
GRE5_2b,42813987,92.38%,3.79%,3.41%,0.27%
GRE6-7_1,69012978,91.15%,4.24%,4.09%,0.33%
Lm-F01Cb,9209878,48.26%,2.59%,20.19%,28.35%
Lm-F02Cb,30385015,56.30%,4.36%,38.77%,0.39%
Lm-F03Cb,26558086,91.67%,4.36%,3.47%,0.32%
Lm-F04Cbf,29020333,91.67%,4.43%,3.42%,0.27%
Lm-F05Cbf,30270985,90.57%,4.38%,4.44%,0.34%
Lm-F06Cbf,25734927,90.51%,3.84%,5.23%,0.23%
Lm-F07b,31145520,84.21%,3.46%,8.05%,4.11%
Lm-F08Cbf,34493366,89.97%,4.68%,4.36%,0.57%
Lm-F09Cf,27682951,88.06%,4.18%,7.40%,0.14%
Lm-F10Cb,34068045,87.15%,3.90%,8.48%,0.25%
Lm-F11b,19259937,86.05%,3.69%,7.76%,2.36%
Lm-F12Cf,30496436,89.24%,4.26%,6.18%,0.13%
Lm-F13Cbf,27470390,88.91%,4.79%,5.66%,0.39%
Lm-F14Cbf,28488534,91.14%,4.20%,4.19%,0.30%
Lm-F15Cbf,23729328,90.86%,3.97%,4.15%,0.83%
Lm-F16b,20485129,80.78%,3.04%,10.55%,5.46%
Lm-M01Cf,27089321,88.46%,4.32%,6.88%,0.13%
Lm-M02Cb,29741982,0.60%,0.04%,99.16%,0.19%
Lm-M03Cb,28771375,0.43%,0.03%,99.37%,0.17%
Lm-M04b,29350846,70.34%,3.58%,11.46%,14.24%
Lm-M05b,37929660,85.43%,3.10%,9.36%,1.97%
Lm-M06Cb,30907785,89.36%,4.09%,5.94%,0.41%
Lm-M07Cf,31448928,90.99%,4.41%,4.11%,0.21%
Lm-M08Cb,29744411,89.00%,4.33%,6.36%,0.13%
Lm-M09b,32074884,81.93%,4.35%,11.45%,2.13%
Lm-M10b,34734030,84.00%,4.35%,6.36%,5.06%
Lm-M11Cb,28940112,86.82%,4.44%,8.00%,0.46%
Lm-M12Cb,35449851,89.12%,4.38%,5.29%,0.95%
Lm-M13Cf,31291298,89.17%,4.31%,6.22%,0.14%
Lm-M14Cb,31940364,88.25%,4.50%,6.30%,0.69%
Lm-M15Cb,37869950,90.14%,4.40%,4.90%,0.39%
OR01F,69321187,92.84%,3.86%,2.94%,0.15%
OR01M,65093878,91.20%,4.21%,4.00%,0.37%
OR02M,53968476,91.23%,3.89%,3.74%,0.87%
OR03Fb,45971855,91.37%,4.00%,4.28%,0.18%
OR03M,55540404,85.93%,3.63%,6.24%,3.87%
OR04F,63669247,91.82%,4.14%,3.46%,0.38%
OR04Mb,51049599,89.48%,3.76%,6.22%,0.38%
OR05F,54386955,91.01%,4.01%,4.26%,0.47%
OR05M,52600002,91.97%,4.03%,3.41%,0.42%
OR06Fb,44902607,91.13%,3.84%,4.63%,0.21%
OR07M,62013234,64.65%,3.87%,13.48%,17.30%
OR08F,63299179,91.38%,3.85%,3.91%,0.62%
OR09F,58298812,91.57%,3.57%,4.33%,0.33%
OR09M,64120730,91.04%,4.14%,3.97%,0.58%
OR10M,76435009,91.33%,4.28%,3.70%,0.53%
TUNI06-M,22259628,67.86%,3.41%,18.72%,9.58%
TUNI10-Mb,38271898,81.13%,3.17%,10.98%,4.46%
TUNI11-Fb,57438553,88.23%,4.60%,5.50%,1.37%
TUNI13-Fb,63952965,80.92%,4.35%,8.79%,5.50%
TUNI14-Fb,56605231,89.69%,4.80%,4.48%,0.79%"""

df = pd.read_csv(pd.io.common.StringIO(data))

for col in ["Unique", "Multi", "Too_short", "Other"]:
    df[col] = df[col].str.replace('%', '').astype(float)

styled_df = (
    df.style
    .background_gradient(subset=["Unique"], cmap="Greens")
    .background_gradient(subset=["Multi"], cmap="Oranges")
    .background_gradient(subset=["Too_short"], cmap="Reds")
    .background_gradient(subset=["Other"], cmap="Reds")
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                    ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]}
    ])
    .format({
        "Unique": "{:.2f}%",
        "Multi": "{:.2f}%",
        "Too_short": "{:.2f}%",
        "Other": "{:.2f}%"
    })
)

styled_df

,Sample,Reads,Unique,Multi,Too_short,Other
0,GRE1-2_5b,58577068,91.08%,4.19%,4.14%,0.38%
1,GRE3-4_1b,56462041,90.49%,3.96%,4.81%,0.58%
2,GRE3-4_2b,54599420,92.13%,4.17%,3.19%,0.32%
3,GRE5_2b,42813987,92.38%,3.79%,3.41%,0.27%
4,GRE6-7_1,69012978,91.15%,4.24%,4.09%,0.33%
5,Lm-F01Cb,9209878,48.26%,2.59%,20.19%,28.35%
6,Lm-F02Cb,30385015,56.30%,4.36%,38.77%,0.39%
7,Lm-F03Cb,26558086,91.67%,4.36%,3.47%,0.32%
8,Lm-F04Cbf,29020333,91.67%,4.43%,3.42%,0.27%
9,Lm-F05Cbf,30270985,90.57%,4.38%,4.44%,0.34%


## 1.4 Results GENOTYPING

#### Script statitstiques VCF: **stats_VCF.sh**

In [ ]:
#!/bin/bash

#SBATCH --job-name=Stats_VCF
#SBATCH -p workq
#SBATCH --time=1-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=8G
#SBATCH --cpus-per-task=4

#############################
# 1. MODULES
#############################

module load devel/python/Python-3.6.3
module load bioinfo/Bcftools/1.9
module load bioinfo/samtools/1.14
module load bioinfo/VCFtools/0.1.16

#############################
# 2. FILES 
#############################

VCF=/home/tbertrand/work/L_nobilis_all/GENOTYPING/STATS_VCF/L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf
OUTPUTDIR=/home/tbertrand/work/L_nobilis_all/GENOTYPING/STATS_VCF
BASENAME=L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2

#############################
# 3. RUN 
#############################

# Compression + indexation
bgzip -c $VCF > ${OUTPUTDIR}/${BASENAME}.vcf.gz
tabix -p vcf ${OUTPUTDIR}/${BASENAME}.vcf.gz

# Utilisation du vcf.gz pour les stats
VCFGZ=${OUTPUTDIR}/${BASENAME}.vcf.gz

# Global stats 
bcftools stats $VCFGZ > ${OUTPUTDIR}/${BASENAME}.global.stats

# Depth per individual 
vcftools --gzvcf $VCFGZ --depth --out ${OUTPUTDIR}/${BASENAME}.depth_indiv

# Missingness
vcftools --gzvcf $VCFGZ --missing-indv --out ${OUTPUTDIR}/${BASENAME}.missing_indiv

# Heterozygosity
vcftools --gzvcf $VCFGZ --het --out ${OUTPUTDIR}/${BASENAME}.het

# Relatedness2
vcftools --gzvcf $VCFGZ --relatedness2 --out ${OUTPUTDIR}/${BASENAME}.relatedness2

#### STATS SNP BCFTOOLS

In [56]:
import pandas as pd

# Données extraites manuellement du fichier
data = [
    ("number of samples", 56),
    ("number of records", 34718369),
    ("number of no-ALTs", 33308303),
    ("number of SNPs", 1376380),
    ("number of MNPs", 44283),
    ("number of indels", 229),
    ("number of others", 16),
    ("number of multiallelic sites", 13457),
    ("number of multiallelic SNP sites", 1803)
]

# Création du DataFrame
df = pd.DataFrame(data, columns=["Metric", "Value"])

styled_df = (
    df.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({"Value": lambda x: f"{x:,}".replace(",", " ")})  )# séparation milliers)

styled_df

df.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/STATS_SNP/bcftools_stats_data_L_nobilis_all.csv", index=False)

#### STATS **MISSING VALUES PER INDIVIDUALS**

In [55]:
import pandas as pd
from io import StringIO

data = """INDV	N_DATA	N_GENOTYPES_FILTERED	N_MISS	F_MISS
Lm-M14Cb	34718369	0	11814641	0.340299
Lm-M11Cb	34718369	0	13030730	0.375327
Lm-M09b	34718369	0	21336723	0.614566
Lm-M02Cb	34718369	0	34270197	0.987091
Lm-F16b	34718369	0	21478318	0.618644
Lm-F15Cbf	34718369	0	18215202	0.524656
Lm-M03Cb	34718369	0	34411012	0.991147
Lm-F13Cbf	34718369	0	13074597	0.37659
Lm-F12Cf	34718369	0	15816660	0.45557
Lm-F09Cf	34718369	0	14518477	0.418179
Lm-F08Cbf	34718369	0	13374744	0.385235
Lm-M15Cb	34718369	0	12435259	0.358175
Lm-F07b	34718369	0	19600818	0.564566
Lm-F05Cbf	34718369	0	14417907	0.415282
OR05F	34718369	0	9521154	0.27424
Lm-M04b	34718369	0	19678571	0.566806
Lm-F02Cb	34718369	0	19468142	0.560745
OR03M	34718369	0	8996374	0.259124
OR09M	34718369	0	8347399	0.240432
Lm-M05b	34718369	0	16292232	0.469268
OR01M	34718369	0	8539135	0.245954
Lm-M13Cf	34718369	0	14344171	0.413158
Lm-M12Cb	34718369	0	14831619	0.427198
Lm-M07Cf	34718369	0	14603771	0.420635
Lm-F04Cbf	34718369	0	14714918	0.423837
Lm-F06Cbf	34718369	0	15631722	0.450244
OR01F	34718369	0	7516384	0.216496
GRE6-7_1	34718369	0	7592060	0.218676
Lm-M10b	34718369	0	17762538	0.511618
Lm-F10Cb	34718369	0	12344761	0.355569
GRE5_2b	34718369	0	8986014	0.258826
OR04Mb	34718369	0	8531696	0.24574
OR05M	34718369	0	8689378	0.250282
TUNI06-M	34718369	0	14669153	0.422518
GRE3-4_1b	34718369	0	8255138	0.237774
Lm-F03Cb	34718369	0	11819025	0.340426
OR03Fb	34718369	0	8758862	0.252283
OR04F	34718369	0	8115724	0.233759
GRE3-4_2b	34718369	0	8291115	0.238811
GRE1-2_5b	34718369	0	7680647	0.221227
Lm-F11b	34718369	0	21851322	0.629388
OR06Fb	34718369	0	8729855	0.251448
OR10M	34718369	0	7577856	0.218266
Lm-M08Cb	34718369	0	14681208	0.422866
OR09F	34718369	0	9169684	0.264116
OR02M	34718369	0	9095621	0.261983
OR07M	34718369	0	10206935	0.293992
Lm-F14Cbf	34718369	0	16638973	0.479256
OR08F	34718369	0	8502728	0.244906
TUNI10-Mb	34718369	0	8736201	0.251631
TUNI13-Fb	34718369	0	6803889	0.195974
TUNI14-Fb	34718369	0	6492436	0.187003
Lm-M06Cb	34718369	0	15784414	0.454642
Lm-M01Cf	34718369	0	14525662	0.418385
TUNI11-Fb	34718369	0	6568335	0.189189
Lm-F01Cb	34718369	0	27083306	0.780086
"""

df = pd.read_csv(StringIO(data), sep="\t")

# Tri par missing décroissant
df = df.sort_values(by="F_MISS", ascending=False)

styled_df = (
    df.style
    .background_gradient(subset=["F_MISS"], cmap="Blues")
    .set_properties(**{
        'font-size': '11pt',
    })
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                    ('text-align', 'center'),]},
        {'selector': 'td', 'props': [('text-align', 'center')]}
    ])
    .format({
        "F_MISS": "{:.3f}",
        "N_MISS": lambda x: f"{x:,.0f}".replace(",", " "),  # sépare par milliers,
        "N_DATA": lambda x: f"{x:,.0f}".replace(",", " "),  # sépare par milliers,
    })
)

styled_df

df.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/STATS_SNP/missing_data_L_nobilis_all.csv", index=False)

#### STATS **MEAN DEPTH PER INDIVIDUALS**

In [54]:
import pandas as pd

data = """INDV	N_SITES	MEAN_DEPTH
Lm-M14Cb	22903728	162.044
Lm-M11Cb	21687639	155.922
Lm-M09b	13381646	287.556
Lm-M02Cb	448172	40.5443
Lm-F16b	13240051	181.531
Lm-F15Cbf	16503167	183.591
Lm-M03Cb	307357	42.8206
Lm-F13Cbf	21643772	139.083
Lm-F12Cf	18901709	182.802
Lm-F09Cf	20199892	156.417
Lm-F08Cbf	21343625	182.686
Lm-M15Cb	22283110	207.916
Lm-F07b	15117551	242.455
Lm-F05Cbf	20300462	181.964
OR05F	25197215	278.985
Lm-M04b	15039798	184.947
Lm-F02Cb	15250227	147.882
OR03M	25721995	257.977
OR09M	26370970	296.067
Lm-M05b	18426137	264.315
OR01M	26179234	320.758
Lm-M13Cf	20374198	178.032
Lm-M12Cb	19886750	222.201
Lm-M07Cf	20114598	187.601
Lm-F04Cbf	20003451	177.832
Lm-F06Cbf	19086647	173.444
OR01F	27201985	327.554
GRE6-7_1	27126309	296.878
Lm-M10b	16955831	239.276
Lm-F10Cb	22373608	186.206
GRE5_2b	25732355	202.371
OR04Mb	26186673	233.012
OR05M	26028991	238.106
TUNI06-M	20049216	87.0024
GRE3-4_1b	26463231	251.604
Lm-F03Cb	22899344	139.584
OR03Fb	25959507	220.884
OR04F	26602645	296.321
GRE3-4_2b	26427254	248.051
GRE1-2_5b	27037722	269.266
Lm-F11b	12867047	181.211
OR06Fb	25988514	203.879
OR10M	27140513	349.631
Lm-M08Cb	20037161	180.42
OR09F	25548685	293.137
OR02M	25622748	266.659
OR07M	24511434	192.446
Lm-F14Cbf	18079396	195.361
OR08F	26215641	299.099
TUNI10-Mb	25982168	161.638
TUNI13-Fb	27914480	232.942
TUNI14-Fb	28225933	230.164
Lm-M06Cb	18933955	199.939
Lm-M01Cf	20192707	150.789
TUNI11-Fb	28150034	239.894
Lm-F01Cb	7635063	85.8592
"""

df = pd.read_csv(pd.io.common.StringIO(data), sep="\t")
df = df.sort_values(by="MEAN_DEPTH", ascending=False)

styled_df = (
    df.style
    .background_gradient(subset=["MEAN_DEPTH"], cmap="Blues")
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                    ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'center')]}
    ])
    .format({
        "N_SITES": lambda x: f"{x:,.0f}".replace(",", " "),  # sépare par milliers,
        "MEAN_DEPTH": "{:.1f}"
    })
)

styled_df

df.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/STATS_SNP/mean_depth_per_indiv_L_nobilis_all.csv", index=False)

# 2. SDPOP

## 2.0 Before running SDpop

0. Create directory

In [ ]:
mkdir SD_POP
mkdir WORKFLOW_SDPOP

1. create file BED_EXON

In [ ]:
# creation BED_EXON
cd /home/tbertrand/work/L_nobilis_all/GENOME

grep "exon" braker.fixed_ids.gff3 \
| awk 'BEGIN{OFS="\t"}{
    match($9,/gene_id=([^;]+)/,a);
    print $1,$4-1,$5,a[1]
}' \
| sort -k1,1 -k2,2n \
> braker.fixed_ids_exons_sorted.bed

2. Select the individulals to filter from the vcf

In [ ]:
# Mauvais mapping
Lm-M02Cb
Lm-M03Cb
Lm-F01Cb

#Pas de sexage
GRE1-2_5b
GRE3-4_1b
GRE3-4_2b
GRE5_2b
GRE6-7_1

3. Adapt the rule pop_sum in order to choose corectly the individual sex depending on the indivudals name

In [ ]:
        INDIV_LIST_FEMALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep -E 'F' | paste -sd ',' -)
        INDIV_LIST_MALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep -E 'M' | paste -sd ',' -)

## 2.1 SDPOP WORKFLOW

#### config.yaml

In [ ]:
WORKDIR_SDPOP: "/home/tbertrand/work/L_nobilis_all/WORKFLOW_SD_POP"
VCF: "/home/tbertrand/work/L_nobilis_all/GENOTYPING/L_nobilis_all_Lazorica_VCFALLSITES_C2_REPMONO_NBESTALLELES2.vcf"
ref_genome: "/home/tbertrand/work/L_nobilis_all/GENOME/Index/GCA_977007225.1_dmLauAzor1.pri_genomic.fna"
EXONS_BED: "/home/tbertrand/work/L_nobilis_all/GENOME/braker.fixed_ids_exons_sorted.bed"
remove_individuals: "/home/tbertrand/work/L_nobilis_all/WORKFLOW_SD_POP/remove_individuals.txt"

VCF_NORM: "/home/tbertrand/work/L_nobilis_all/SD_POP/L_nobilis_all_Lazorica_norm.vcf"
VCF_NORM_FILT: "/home/tbertrand/work/L_nobilis_all/SD_POP/L_nobilis_all_Lazorica_norm_filtered.vcf"
VCF_NORM_FILT_EXONS_RECODE: "/home/tbertrand/work/L_nobilis_all/SD_POP/L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.vcf"
CNT_FILE: "/home/tbertrand/work/L_nobilis_all/SD_POP/geno_count_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"

SDPOP_NOSEX_OUT: "/home/tbertrand/work/L_nobilis_all/SD_POP/SDPOP_nosexchr_L_nobilis_all_Lazorica_norm.vcf_norm_filtered_EXONS.recode.out"
SDPOP_XY_OUT: "/home/tbertrand/work/L_nobilis_all/SD_POP/SDPOP_XY_L_nobilis_all_Lazorica_norm.vcf_norm_filtered_EXONS.recode.out"
SDPOP_ZW_OUT: "/home/tbertrand/work/L_nobilis_all/SD_POP/SDPOP_ZW_L_nobilis_all_Lazorica_norm.vcf_norm_filtered_EXONS.recode.out"

NOSEX_LOG: "iteration_no_sex_chr_Lnob_all_Lazo.txt"
XY_LOG: "iteration_XY_sex_chr_Lnob_all_Lazo.txt"
ZW_LOG: "iteration_ZW_sex_chr_Lnob_all_Lazo.txt"

#### SDPOP snakefile

In [ ]:
# L objectif de ce script est prendre en INNPUT un fichier VCF brut (freebayes), de le formater pour SDpop et run SDpop

#######################################
# 1. CONFIG
#######################################

configfile: "config.yaml"

WORKDIR_SDPOP = config["WORKDIR_SDPOP"]
VCF           = config["VCF"]
ref_genome    = config["ref_genome"]
EXONS_BED     = config["EXONS_BED"]

VCF_NORM       = config["VCF_NORM"]
VCF_NORM_FILT  = config["VCF_NORM_FILT"]
VCF_NORM_FILT_EXONS_RECODE = config["VCF_NORM_FILT_EXONS_RECODE"]
CNT_FILE       = config["CNT_FILE"]

SDPOP_NOSEX_OUT = config["SDPOP_NOSEX_OUT"]
SDPOP_XY_OUT    = config["SDPOP_XY_OUT"]
SDPOP_ZW_OUT    = config["SDPOP_ZW_OUT"]

NOSEX_LOG = config["NOSEX_LOG"]
XY_LOG    = config["XY_LOG"]
ZW_LOG    = config["ZW_LOG"]


###########################################
# CIBLES FINALES 
###########################################

rule all:
    input:
        VCF_NORM,
        VCF_NORM_FILT,
        VCF_NORM_FILT_EXONS_RECODE,
        CNT_FILE,
        SDPOP_NOSEX_OUT,
        SDPOP_XY_OUT,
        SDPOP_ZW_OUT,
        NOSEX_LOG,
        XY_LOG,
        ZW_LOG


###############################################
# 3. RUN BCFTOOLS norm 
###############################################

rule bcftools_norm:
    input:
        vcf = VCF,
        ref = ref_genome
    output:
        temp(VCF_NORM)
    threads: 6
    resources:
        mem_mb=48000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/Bcftools/1.9

        bcftools norm --threads {threads} {input.vcf} -f {input.ref} -m +any -O v -o {output}
        """


##################################################
# 4. RUN FILTER VCFTOOLS 
##################################################
#Ici on filtre la profondeur minimal a 7 read par individus et on retire l individu Lm-F01Cb

rule vcftools_filter:
    input:
        VCF_NORM=VCF_NORM,
        remove_individuals=config["remove_individuals"]
    output:
        temp(VCF_NORM_FILT)
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/samtools/1.14
        module load bioinfo/VCFtools/0.1.16

        vcftools --vcf {input.VCF_NORM} --minDP 7 --remove {input.remove_individuals} --recode --out {output}
        mv {output}.recode.vcf {output}
        """


#########################################################
# 5. CUT EXONS WINDOWS 
#########################################################

rule exons_intersect:
    input:
        vcf = VCF_NORM_FILT,
        bed = EXONS_BED
    output:
        VCF_NORM_FILT_EXONS_RECODE
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """

        module load devel/python/Python-3.11.1
        module load bioinfo/bedtools/2.30.0

        # extract the header line (CHROM POS ...) of the vcf
        grep -m 1 "CHROM" "{input.vcf}" > "{output}"

        # create an intersect of the vcf and the bed file
        bedtools intersect -a "{input.vcf}" -b "{input.bed}" -wa -wb | \
        awk '{{
            ## We want to retrieve here the last column which corresponds to the name of the gene
            gene_id = $NF
            sub(/-.*/, "", gene_id)

            # print gene_id as the first column 
            printf "%s\t", gene_id

            # Print all the VCF column except the first (#CHROM) and the last 4 (BED column)
            for(i=2;i<=NF-4;i++) printf "%s\t", $i

            print ""
        }}' | sort -k1,1V -k2,2n | uniq >> "{output}"
        # The sort is in order to guarantees one continuous block per gene.
        """


#############################################
# 6. RUN POPSUM 
#############################################

rule popsum_counts:
    input:
        VCF_NORM_FILT_EXONS_RECODE
    output:
        CNT_FILE
    threads: 1
    resources:
        mem_mb=8000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148

        INDIV_LIST_FEMALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep -E 'F' | paste -sd ',' -)
        INDIV_LIST_MALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep -E 'M' | paste -sd ',' -)

        popsum "{input}" "{output}" v "$INDIV_LIST_FEMALES" "$INDIV_LIST_MALES"
        """


#############################################
# 7. RUN SDPOP 
#############################################

rule sdpop_nosex:
    input:
        CNT_FILE
    output:
        SDPOP_NOSEX_OUT
    log:
        NOSEX_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e n 1 1 s > "{log}"
        """

rule sdpop_xy:
    input:
        CNT_FILE
    output:
        SDPOP_XY_OUT
    log:
        XY_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e x 1 1 s > "{log}"
        """

rule sdpop_zw:
    input:
        CNT_FILE
    output:
        SDPOP_ZW_OUT
    log:
        ZW_LOG
    threads: 1
    resources:
        mem_mb=16000,
        time="1-00:00:00"
    shell:
        """
        module load bioinfo/SDpop/f112148
        sdpop "{input}" "{output}" e z 1 1 s > "{log}"
        """

#### run_snakefile_SDpop

In [ ]:
#!/bin/bash
#SBATCH --job-name=Snakemake_SDPOP_TEST_Lnobilis_L_azorica
#SBATCH -p unlimitq
#SBATCH --time=2-00:00:00
#SBATCH --mail-user=tristan.bertrand@umontpellier.fr
#SBATCH --mail-type=ALL
#SBATCH --cpus-per-task=1
#SBATCH --mem=2G
#SBATCH --output=snakemake_%j.log

module load bioinfo/Snakemake/8.20.3

snakemake \
  --snakefile snakefile_SD_POP \
  --cores 6 \
  --rerun-incomplete \
  --keep-going \
  --latency-wait 60 \
  --executor slurm \
  --jobs 4

## 2.2 SDPOP result

### Import the results

In [ ]:
# Import results Nobilis
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP/SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP/SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP/SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP


In [ ]:
#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

## Result models SDPOP ALL (SDPOP_1)

#ZW
pi 0.952190  0.029141  0.003323  0.014127  0.001220 ; rho 0.498937  0.001063 ; e0: 8.111875e-05, log-likelihood: -10370292.489761, BIC (sites): 20740666.113587, BIC (contigs): 20740645.086578, BIC (sites*individuals): 20740687.415512

#XY
pi 0.959657  0.031057  0.003371  0.005830  0.000085 ; rho 0.334768  0.165232 ; e0: 9.342905e-05, log-likelihood: -10372741.557958, BIC (sites): 20745564.249982, BIC (contigs): 20745543.222972, BIC (sites*individuals): 20745585.551906

#NOSEX
pi 0.964123  0.032469  0.003408 ; rho; e0: 1.100720e-04, log-likelihood: -10373770.104787, BIC (sites): 20747580.776607, BIC (contigs): 20747570.263102, BIC (sites*individuals): 20747591.427569

In [ ]:
import pandas as pd

data = [
    {
        "Model": "XY",
        "pi": "0.959657, 0.031057, 0.003371, 0.005830, 0.000085",
        "rho": "0.334768, 0.165232",
        "e0": 9.342905e-05,
        "log_likelihood": -10372741.557958,
        "BIC_sites": 20745564.249982,
        "BIC_contigs": 20745543.222972,
        "BIC_sites_ind": 20745585.551906
    },
    {
        "Model": "ZW",
        "pi": "0.952190, 0.029141, 0.003323, 0.014127, 0.001220",
        "rho": "0.498937, 0.001063",
        "e0": 8.111875e-05,
        "log_likelihood": -10370292.489761,
        "BIC_sites": 20740666.113587,
        "BIC_contigs": 20740645.086578,
        "BIC_sites_ind": 20740687.415512
    },
    {
        "Model": "NOSEX",
        "pi": "0.964123, 0.032469, 0.003408",
        "rho": "",
        "e0": 1.100720e-04,
        "log_likelihood": -10373770.104787,
        "BIC_sites": 20747580.776607,
        "BIC_contigs": 20747570.263102,
        "BIC_sites_ind": 20747591.427569
    }
]

# Création du DataFrame
df = pd.DataFrame(data)

# Tri par log-likelihood décroissant
df = df.sort_values(by="log_likelihood", ascending=False).reset_index(drop=True)

# Style pour affichage
styled_df = (
    df.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
        "e0": "{:.6e}",
        "log_likelihood": "{:,.3f}",
        "BIC_sites": "{:,.3f}",
        "BIC_contigs": "{:,.3f}",
        "BIC_sites_ind": "{:,.3f}"
    })
)

styled_df

,Model,pi,rho,e0,log_likelihood,BIC_sites,BIC_contigs,BIC_sites_ind
0,ZW,"0.952190, 0.029141, 0.003323, 0.014127, 0.001220","0.498937, 0.001063",8.111875e-05,"-10,370,292.490","20,740,666.114","20,740,645.087","20,740,687.416"
1,XY,"0.959657, 0.031057, 0.003371, 0.005830, 0.000085","0.334768, 0.165232",9.342905e-05,"-10,372,741.558","20,745,564.250","20,745,543.223","20,745,585.552"
2,NOSEX,"0.964123, 0.032469, 0.003408",,1.100720e-04,"-10,373,770.105","20,747,580.777","20,747,570.263","20,747,591.428"


## Result models SDPOP ITA (SDPOP_2)

In [ ]:
# Import results Nobilis SARDINIA
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_2/SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_2
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_2/SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_2
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_2/SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_2

#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

#XY
pi 0.766146  0.006171  0.169514  0.000000  0.058168 ; rho 0.276985  0.223015 ; e0: 5.065271e-05, log-likelihood: -2834982.543289, BIC (sites): 5670041.763297, BIC (contigs): 5670024.275682, BIC (sites*individuals): 5670057.268387

#ZW
pi 0.655729  0.006061  0.148173  0.000001  0.190037 ; rho 0.492308  0.007692 ; e0: 2.511854e-04, log-likelihood: -2816021.473040, BIC (sites): 5632119.622799, BIC (contigs): 5632102.135183, BIC (sites*individuals): 5632135.127888

#no sex
pi 0.822331  0.006035  0.171634 ; rho; e0: 4.694102e-05, log-likelihood: -2842939.253866, BIC (sites): 5685916.846091, BIC (contigs): 5685908.102284, BIC (sites*individuals): 5685924.598636


In [39]:
import pandas as pd
data = [
    {
        "Model": "XY",
        "pi": "0.766146, 0.006171, 0.169514, 0.000000, 0.058168",
        "rho": "0.276985, 0.223015",
        "e0": 5.065271e-05,
        "log_likelihood": -2834982.543289,
        "BIC_sites": 5670041.763297,
        "BIC_contigs": 5670024.275682,
        "BIC_sites_ind": 5670057.268387
    },
    {
        "Model": "ZW",
        "pi": "0.655729, 0.006061, 0.148173, 0.000001, 0.190037",
        "rho": "0.492308, 0.007692",
        "e0": 2.511854e-04,
        "log_likelihood": -2816021.473040,
        "BIC_sites": 5632119.622799,
        "BIC_contigs": 5632102.135183,
        "BIC_sites_ind": 5632135.127888
    },
    {
        "Model": "NOSEX",
        "pi": "0.822331, 0.006035, 0.171634",
        "rho": "",
        "e0": 4.694102e-05,
        "log_likelihood": -2842939.253866,
        "BIC_sites": 5685916.846091,
        "BIC_contigs": 5685908.102284,
        "BIC_sites_ind": 5685924.598636
    }
]

# Création du DataFrame
df = pd.DataFrame(data)

# Tri par log-likelihood décroissant
df = df.sort_values(by="log_likelihood", ascending=False).reset_index(drop=True)

# Style pour affichage
styled_df = (
    df.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
        "e0": "{:.6e}",
        "log_likelihood": "{:,.3f}",
        "BIC_sites": "{:,.3f}",
        "BIC_contigs": "{:,.3f}",
        "BIC_sites_ind": "{:,.3f}"
    })
)

styled_df

,Model,pi,rho,e0,log_likelihood,BIC_sites,BIC_contigs,BIC_sites_ind
0,ZW,"0.655729, 0.006061, 0.148173, 0.000001, 0.190037","0.492308, 0.007692",2.511854e-04,"-2,816,021.473","5,632,119.623","5,632,102.135","5,632,135.128"
1,XY,"0.766146, 0.006171, 0.169514, 0.000000, 0.058168","0.276985, 0.223015",5.065271e-05,"-2,834,982.543","5,670,041.763","5,670,024.276","5,670,057.268"
2,NOSEX,"0.822331, 0.006035, 0.171634",,4.694102e-05,"-2,842,939.254","5,685,916.846","5,685,908.102","5,685,924.599"


## SD POP 3 (POR)

In [ ]:
INDIV_LIST_FEMALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep -E '(^|[-_])F' | paste -sd ',' -)
INDIV_LIST_MALES=$(grep "^#CHROM" "{input}" | cut -f10- | tr '\t' '\n' | grep -E '(^|[-_])M' | paste -sd ',' -)

In [ ]:
# Import results Nobilis TUN resex
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_3/SDPOP_XY_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_3
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_3/SDPOP_ZW_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_3
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_3/SDPOP_nosexchr_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_3

#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_POR_Lazorica_norm_filtered_EXONS.recode.out.contig

## SD POP 4 (All indiv resex -GRE)

In [ ]:
INDIV_LIST_FEMALES="Lm-F16b,Lm-,Lm-F13Cbf,Lm-F12Cf,Lm-F09Cf,Lm-F08Cbf,Lm-F07b,Lm-F05Cbf,OR05F,Lm-F02Cb,Lm-F04Cbf,Lm-F06Cbf,OR01F,Lm-F10Cb,Lm-F03Cb,Lm-F14Cbf,Lm-F15Cbf,Lm-F11b,OR03Fb,OR04F,OR06Fb,OR09F,OR08F,OR03M,OR09M,OR02M,OR04Mb,OR05M,Lm-F01Cb,TUNI06-M,TUNI10-Mb"

INDIV_LIST_MALES="Lm-M14Cb,Lm-M11Cb,Lm-M09b,Lm-M15Cb,Lm-M04b,Lm-M05b,Lm-M13Cf,Lm-M12Cb,Lm-M07Cf,Lm-M10b,Lm-M08Cb,Lm-M06Cb,Lm-M01Cf,OR07M,OR10M,OR01M,TUNI13-Fb,TUNI14-Fb,TUNI11-Fb"  

In [ ]:
# Import results Nobilis ALL resex
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_4/SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_4
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_4/SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_4
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_4/SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_4

#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

XY
pi 0.954640  0.029306  0.003473  0.011390  0.001192 ; rho 0.351944  0.148056 ; e0: 8.518736e-05, log-likelihood: -9916319.676032, BIC (sites): 19832720.366069, BIC (contigs): 19832699.416152, BIC (sites*individuals): 19832741.440348

ZW
pi 0.955108  0.028974  0.003346  0.008831  0.003741 ; rho 0.425994  0.074006 ; e0: 7.572764e-05, log-likelihood: -9916470.791642, BIC (sites): 19833022.597289, BIC (contigs): 19833001.647372, BIC (sites*individuals): 19833043.671568

nosex
pi 0.965005  0.031493  0.003503 ; rho; e0: 1.007110e-04, log-likelihood: -9919328.119488, BIC (sites): 19838696.745978, BIC (contigs): 19838686.271020, BIC (sites*individuals): 19838707.283117


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Données SDpop
data = [
    {
        "Model": "XY",
        "autosomal.pr": 0.954640,
        "haploid.pr": 0.029306,
        "paralogs.pr": 0.003473,
        "hemizygote.pr": 0.011390,
        "sexlinked.pr": 0.001192,
        "rho": "0.351944, 0.148056",
        "e0": 8.518736e-05,
        "log_likelihood": -9916319.676032,
        "BIC_sites": 19832720.366069,
        "BIC_contigs": 19832699.416152,
        "BIC_sites_ind": 19832741.440348
    },
    {
        "Model": "ZW",
        "autosomal.pr": 0.955108,
        "haploid.pr": 0.028974,
        "paralogs.pr": 0.003346,
        "hemizygote.pr": 0.008831,
        "sexlinked.pr": 0.003741,
        "rho": "0.425994, 0.074006",
        "e0": 7.572764e-05,
        "log_likelihood": -9916470.791642,
        "BIC_sites": 19833022.597289,
        "BIC_contigs": 19833001.647372,
        "BIC_sites_ind": 19833043.671568
    },
    {
        "Model": "NOSEX",
        "autosomal.pr": 0.965005,
        "haploid.pr": 0.031493,
        "paralogs.pr": 0.003503,
        "hemizygote.pr": np.nan,
        "sexlinked.pr": np.nan,
        "rho": "",
        "e0": 1.007110e-04,
        "log_likelihood": -9919328.119488,
        "BIC_sites": 19838696.745978,
        "BIC_contigs": 19838686.271020,
        "BIC_sites_ind": 19838707.283117
    }
]

# DataFrame
df = pd.DataFrame(data)

# Tri par log-likelihood
df = df.sort_values(by="log_likelihood", ascending=False).reset_index(drop=True)

# Tableau 1 : stats modèles
df_stats = df[[
    "Model",
    "log_likelihood",
    "BIC_sites",
    "BIC_contigs",
    "BIC_sites_ind"
]]

styled_stats = (
    df_stats.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
        "log_likelihood": "{:,.3f}",
        "BIC_sites": "{:,.3f}",
        "BIC_contigs": "{:,.3f}",
        "BIC_sites_ind": "{:,.3f}"
    })
)

# Tableau 2 : priors
df_priors = df[[
    "Model",
    "autosomal.pr",
    "haploid.pr",
    "paralogs.pr",
    "hemizygote.pr",
    "sexlinked.pr"
]]

styled_priors = (
    df_priors.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
        "autosomal.pr": "{:.6f}",
        "haploid.pr": "{:.6f}",
        "paralogs.pr": "{:.6f}",
        "hemizygote.pr": "{:.6f}",
        "sexlinked.pr": "{:.6f}"
    })
)

# Affichage
display(Markdown("### Model comparison"))
display(styled_stats)

display(Markdown("### Priors"))
display(styled_priors)

df_stats.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/results_15_04/model_nobilis_stats.csv", index=False)
df_priors.to_csv("/Users/cibio/Desktop/Laurus/SDPOP/results_15_04/model_nobilis_priors.csv", index=False)

### Model comparison

,Model,log_likelihood,BIC_sites,BIC_contigs,BIC_sites_ind
0,XY,"-9,916,319.676","19,832,720.366","19,832,699.416","19,832,741.440"
1,ZW,"-9,916,470.792","19,833,022.597","19,833,001.647","19,833,043.672"
2,NOSEX,"-9,919,328.119","19,838,696.746","19,838,686.271","19,838,707.283"


### Priors

,Model,autosomal.pr,haploid.pr,paralogs.pr,hemizygote.pr,sexlinked.pr
0,XY,0.954640,0.029306,0.003473,0.011390,0.001192
1,ZW,0.955108,0.028974,0.003346,0.008831,0.003741
2,NOSEX,0.965005,0.031493,0.003503,nan,nan


## SD POP 5 (ITA resex)

In [ ]:
INDIV_LIST_FEMALES="OR05F,OR01F,OR03Fb,OR04F,OR06Fb,OR09F,OR08F,OR03M,OR09M,OR02M,OR04Mb,OR05M"
INDIV_LIST_MALES="OR07M,OR10M,OR01M"

In [ ]:
# Import results Nobilis ITA ALL resex
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_5/SDPOP_XY_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_5
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_5/SDPOP_ZW_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_5
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_5/SDPOP_nosexchr_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_5

#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_ITA_Lazorica_norm_filtered_EXONS.recode.out.contig

## SD POP 6 (TUN resex)

In [ ]:
INDIV_LIST_FEMALES="TUNI06-M,TUNI10-Mb"
INDIV_LIST_MALES="TUNI11-Fb,TUNI13-Fb,TUNI14-Fb"

In [ ]:
# Import results Nobilis TUN resex
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_6/SDPOP_XY_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_6
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_6/SDPOP_ZW_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_6
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_6/SDPOP_nosexchr_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_6

#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_TUN_Lazorica_norm_filtered_EXONS.recode.out.contig

## SD POP 7 (GRE resex)

In [ ]:
INDIV_LIST_FEMALES="GRE6-7_1,GRE3-4_2b"
INDIV_LIST_MALES="GRE5_2b,GRE1-2_5b,GRE3-4_1b"

In [ ]:
# Import results Nobilis GRE resex
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_7/SDPOP_XY_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_7
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_7/SDPOP_ZW_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_7
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_7/SDPOP_nosexchr_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_7

#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_GRE_Lazorica_norm_filtered_EXONS.recode.out.contig

## SD POP 8 (All indiv resex +GRE)

In [ ]:
INDIV_LIST_FEMALES="Lm-F16b,Lm-,Lm-F13Cbf,Lm-F12Cf,Lm-F09Cf,Lm-F08Cbf,Lm-F07b,Lm-F05Cbf,OR05F,Lm-F02Cb,Lm-F04Cbf,Lm-F06Cbf,OR01F,Lm-F10Cb,Lm-F03Cb,Lm-F14Cbf,Lm-F15Cbf,Lm-F11b,OR03Fb,OR04F,OR06Fb,OR09F,OR08F,OR03M,OR09M,OR02M,OR04Mb,OR05M,Lm-F01Cb,TUNI06-M,TUNI10-Mb,GRE6-7_1,GRE3-4_2b"

INDIV_LIST_MALES="Lm-M14Cb,Lm-M11Cb,Lm-M09b,Lm-M15Cb,Lm-M04b,Lm-M05b,Lm-M13Cf,Lm-M12Cb,Lm-M07Cf,Lm-M10b,Lm-M08Cb,Lm-M06Cb,Lm-M01Cf,OR07M,OR10M,OR01M,TUNI13-Fb,TUNI14-Fb,TUNI11-Fb,GRE5_2b,GRE1-2_5b,GRE3-4_1b"  

In [ ]:
# Import results Nobilis ALL resex
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_8/SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_8
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_8/SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_8
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_8/SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_8

#L_nobilis_all
grep '>' SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_ZW_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

grep '>' SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig
sed -i -e "s/ /\t/g" SDPOP_nosexchr_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

XY
pi 0.961374  0.028595  0.002120  0.006978  0.000933 ; rho 0.287175  0.212825 ; e0: 9.008292e-05, log-likelihood: -12200463.683547, BIC (sites): 24401009.066355, BIC (contigs): 24400987.502978, BIC (sites*individuals): 24401031.014613

ZW
pi 0.958806  0.027169  0.002065  0.010548  0.001412 ; rho 0.448139  0.051861 ; e0: 7.428488e-05, log-likelihood: -12200224.287211, BIC (sites): 24400530.273685, BIC (contigs): 24400508.710307, BIC (sites*individuals): 24400552.221942

nosex
pi 0.967837  0.030020  0.002143 ; rho; e0: 1.037241e-04, log-likelihood: -12203754.871472, BIC (sites): 24407550.592574, BIC (contigs): 24407539.810885, BIC (sites*individuals): 24407561.566703

In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display, Markdown

# Données SDpop
data = [
    {
        "Model": "XY",
        "autosomal.pr": 0.961374,
        "haploid.pr": 0.028595,
        "paralogs.pr": 0.002120,
        "hemizygote.pr": 0.006978,
        "sexlinked.pr": 0.000933,
        "rho": "0.287175, 0.212825",
        "e0": 9.008292e-05,
        "log_likelihood": -12200463.683547,
        "BIC_sites": 24401009.066355,
        "BIC_contigs": 24400987.502978,
        "BIC_sites_ind": 24401031.014613
    },
    {
        "Model": "ZW",
        "autosomal.pr": 0.958806,
        "haploid.pr": 0.027169,
        "paralogs.pr": 0.002065,
        "hemizygote.pr": 0.010548,
        "sexlinked.pr": 0.001412,
        "rho": "0.448139, 0.051861",
        "e0": 7.428488e-05,
        "log_likelihood": -12200224.287211,
        "BIC_sites": 24400530.273685,
        "BIC_contigs": 24400508.710307,
        "BIC_sites_ind": 24400552.221942
    },
    {
        "Model": "NOSEX",
        "autosomal.pr": 0.967837,
        "haploid.pr": 0.030020,
        "paralogs.pr": 0.002143,
        "hemizygote.pr": 0,
        "sexlinked.pr": 0,
        "rho": "",
        "e0": 1.037241e-04,
        "log_likelihood": -12203754.871472,
        "BIC_sites": 24407550.592574,
        "BIC_contigs": 24407539.810885,
        "BIC_sites_ind": 24407561.566703
    }
]

df = pd.DataFrame(data)

# Tri par log-likelihood
df = df.sort_values(by="log_likelihood", ascending=False).reset_index(drop=True)

# Tableau 1 : stats modèles

df_stats = df[[
    "Model",
    "log_likelihood",
    "BIC_sites",
    "BIC_contigs",
    "BIC_sites_ind"
]]

styled_stats = (
    df_stats.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
        "log_likelihood": "{:,.3f}",
        "BIC_sites": "{:,.3f}",
        "BIC_contigs": "{:,.3f}",
        "BIC_sites_ind": "{:,.3f}"
    })
)

# Tableau 2 : priors

df_priors = df[[
    "Model",
    "autosomal.pr",
    "haploid.pr",
    "paralogs.pr",
    "hemizygote.pr",
    "sexlinked.pr"
]]

styled_priors = (
    df_priors.style
    .set_properties(**{'font-size': '11pt'})
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '12pt'),
                                     ('text-align', 'center')]},
        {'selector': 'td', 'props': [('text-align', 'left')]}
    ])
    .format({
        "autosomal.pr": "{:.6f}",
        "haploid.pr": "{:.6f}",
        "paralogs.pr": "{:.6f}",
        "hemizygote.pr": "{:.6f}",
        "sexlinked.pr": "{:.6f}"
    })
)

# Affichage
display(Markdown("### Model comparison"))
display(styled_stats)

display(Markdown("### Priors"))
display(styled_priors)

### Model comparison

,Model,log_likelihood,BIC_sites,BIC_contigs,BIC_sites_ind
0,ZW,"-12,200,224.287","24,400,530.274","24,400,508.710","24,400,552.222"
1,XY,"-12,200,463.684","24,401,009.066","24,400,987.503","24,401,031.015"
2,NOSEX,"-12,203,754.871","24,407,550.593","24,407,539.811","24,407,561.567"


### Priors

,Model,autosomal.pr,haploid.pr,paralogs.pr,hemizygote.pr,sexlinked.pr
0,ZW,0.958806,0.027169,0.002065,0.010548,0.001412
1,XY,0.961374,0.028595,0.002120,0.006978,0.000933
2,NOSEX,0.967837,0.030020,0.002143,0.000000,0.000000


## 2.4 Gametologs consensus sequence SDPOP_4 (**WZYZgenotyper**)

In [ ]:
#!/bin/bash

#SBATCH --job-name=WZYZ_genotyping_nobilis_all
#SBATCH -p workq
#SBATCH --time=1-00:00:00
#SBATCH --mail-user=tristan.bertrand@supagro.fr
#SBATCH --mail-type=ALL
#SBATCH --mem-per-cpu=10G
#SBATCH --cpus-per-task=1

# This script is for the creation of consensus XY ZW haplotypes fro the genes infered as XY or ZW by SDPOP
# documentation: https://gitlab.in2p3.fr/sex-det-family/sdpop 

########################
# 0. MODULES
########################

module load bioinfo/SDpop/f112148

########################
# 1. INPUT 
########################

#path to the output file of SDPOP
SDPOP_FILE=/home/tbertrand/work/L_nobilis_all/SD_POP_4/SDPOP_XY_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out

#path to the gvcf/vcf file
GVCF=/home/tbertrand/work/L_nobilis_all/SD_POP_4/L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.vcf.gz
VCF=/home/tbertrand/work/L_nobilis_all/SD_POP_4/L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.vcf

#output path
OUTPUT=/home/tbertrand/work/L_nobilis_all/SD_POP_4/wXYz_allgenes_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out

########################
# 2. RUN
########################

gunzip -c "$GVCF" > "$VCF"

# v indicate wxyz_genotyper that we work on a vcf file
# wxyz_....out output file
# 0 : XY/ZW probability threshold: consensus sequences will be produced for any gene with a probability of being XY/ZW above the threshold (here 0)
# 0.95: alleles with a frequency above this threshold will be considered fixed (here 0.95)
# 0.6: alleles with a frequency above this threshold but below the threshold above will be considered not fixed and printed in lowercase letters.
# noprior_xy: work on no prior estimation of frequencies of x and y alleles
# XY: X and Y frequencies should be used
# mean: frequencies averaged over all subtypes should be used

wxyz_genotyper "$SDPOP_FILE" "$VCF" v "$OUTPUT" 0.8 0.95 0.6 noprior_xy XY mean

rm "$VCF"

In [ ]:
scp  tbertrand@genobioinfo2.toulouse.inrae.fr:/home/tbertrand/work/L_nobilis_all/SD_POP_4/wXYz_allgenes_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out /Users/cibio/Desktop/Laurus/SDPOP/L_nobilis_all_L_azorica/SDPOP_4

grep -E '^#|^>' wXYz_allgenes_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out > wXYz_allgenes_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out.contig

# Phylogenie XY

In [ ]:
conda create -n phylo_sexlinked -c bioconda -c conda-forge \
blast mafft trimal iqtree seqkit -y

conda activate phylo_sexlinked

### SCRIPT BLASTP to XY Phylogeny

In [ ]:
#!/bin/bash

############################
# 1. DIRECTORIES
############################

DB_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/Blast_prot_db"
OUT_DIR="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes"
FAA_DIR="$OUT_DIR/faa_prot"
BLAST_DIR="$OUT_DIR/blast"
PERSEA_BLAST_DIR="$BLAST_DIR/persea"
CINNA_BLAST_DIR="$BLAST_DIR/cinna"

mkdir -p "$DB_DIR" "$FAA_DIR" "$PERSEA_BLAST_DIR" "$CINNA_BLAST_DIR"

############################
# 2. INPUT FILES
############################

# Proteins
PERSEA_PROT="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/protein.faa"
CINNA_PROT="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/protein.faa"
# CDS
PERSEA_CDS="/Users/cibio/Desktop/Laurus/genome/Persea_americana/ncbi_dataset/ncbi_dataset/data/GCA_051132755.1/cds_from_genomic.fna"
CINNA_CDS="/Users/cibio/Desktop/Laurus/genome/Cinnamomum_micranthum/ncbi_dataset_Cinnamomum_micranthum/ncbi_dataset/data/GCA_003546025.1/cds_from_genomic.fna"
# Query proteins
PROTEIN_FILE="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/braker.aa"
# Gametologs files 
WXYZ_nobilis="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_nobilis_all_Lazorica_norm_filtered_EXONS.recode.out"
WXYZ_novocanariensis="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_novocanariensis_Lazorica_norm_filtered_EXONS.recode.out"
WXYZ_azorica="/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/wXYz_allgenes_L_azorica_Lazorica_norm_filtered_EXONS.recode.out"
#Annotation file
ANNOTATION="/Users/cibio/Desktop/Laurus/Annotation/Annotation_final/braker.fixed_ids.gtf"

############################
# 1. STRAND FILE (PERMANENT)
############################

STRAND_FILE="$OUT_DIR/gene_strand.txt"

awk '
$3=="gene" {
    print $9 "\t" $7
}' "$ANNOTATION" > "$STRAND_FILE"

############################
# 3. BLAST DATABASES
############################

makeblastdb -in "$PERSEA_PROT" -dbtype prot -out "$DB_DIR/Persea_db"
makeblastdb -in "$CINNA_PROT" -dbtype prot -out "$DB_DIR/Cinna_db"

############################
# 4. GENES LIST
############################

genes=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)

############################
# 5. EXTRACTION + BLAST
############################

for g in "${genes[@]}"
do
    echo "Processing ${g}.t1"

    QUERY="$FAA_DIR/${g}.t1.faa"
    # Extraction
    awk -v gene=">${g}.t1" '
    $0 == gene {print; flag=1; next}
    /^>/ {flag=0}
    flag==1 {print}
    ' "$PROTEIN_FILE" > "$QUERY"

    # BLAST Persea
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Persea_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$PERSEA_BLAST_DIR/${g}_Persea.tsv"

    # BLAST Cinnamomum
    blastp \
    -query "$QUERY" \
    -db "$DB_DIR/Cinna_db" \
    -evalue 1e-4 \
    -max_target_seqs 10 \
    -outfmt "6 qseqid sseqid pident length qlen slen evalue bitscore" \
    -out "$CINNA_BLAST_DIR/${g}_Cinna.tsv"

done


############################
# 6. CDS + BEST HIT DIRECT PAR GENE
############################

for g in "${genes[@]}"
do
    echo "CDS extraction $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    mkdir -p "$GENE_DIR"

    #################################
    # BEST HITS (≥95% IDENTITY)
    #################################

    best_persea=$(awk '$3 >= 95' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | sort -k8,8nr | head -1 | cut -f2)
    best_cinna=$(awk '$3 >= 95' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | sort -k8,8nr | head -1 | cut -f2)

    echo -e "$g  PERSEA:$best_persea  CINNA:$best_cinna"

    #################################
    # SKIP IF NO OUTGROUP AT ALL
    #################################

    if [[ -z "$best_persea" && -z "$best_cinna" ]]; then
        echo "SKIP $g : no outgroup hit ≥95%"
        continue
    fi

    #################################
    # PERSEA CDS (if exists)
    #################################

    if [[ -n "$best_persea" ]]; then
        awk -v id="$best_persea" -v name="${g}_Persea_americana" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$PERSEA_CDS" > "$GENE_DIR/${g}_persea_CDS.fasta"
    else
        echo "WARNING $g : no Persea hit ≥95%"
    fi

    #################################
    # CINNA CDS (if exists)
    #################################

    if [[ -n "$best_cinna" ]]; then
        awk -v id="$best_cinna" -v name="${g}_Cinnamomum_micranthum" '
        /^>/{
            keep=0
            if($0 ~ "protein_id="id){
                keep=1
                print ">"name
                next
            }
        }
        keep && !/^>/ {print}
        ' "$CINNA_CDS" > "$GENE_DIR/${g}_cinna_CDS.fasta"
    else
        echo "WARNING $g : no Cinna hit ≥95%"
    fi

done

############################
# 7. GAMETOLOGS X&Y PAR GENE
############################

for g in "${genes[@]}"
do
    echo "Processing $g"
    GENE_DIR="$OUT_DIR/Gene_familly/$g"
    strand=$(grep -w "^$g" "$STRAND_FILE" | cut -f2)

    #########################
    # NOBILIS
    #########################

    grep -A 1 "${g}_X" "$WXYZ_nobilis" | \
    awk -v name="${g}_nobilis_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_nobilis_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_nobilis" | \
    awk -v name="${g}_nobilis_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_nobilis_Y.fasta"

    #########################
    # NOVOCANARIENSIS
    #########################

    grep -A 1 "${g}_X" "$WXYZ_novocanariensis" | \
    awk -v name="${g}_novocanariensis_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_novocanariensis_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_novocanariensis" | \
    awk -v name="${g}_novocanariensis_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_novocanariensis_Y.fasta"

    #########################
    # AZORICA
    #########################

    grep -A 1 "${g}_X" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_X" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_X.fasta"

    grep -A 1 "${g}_Y" "$WXYZ_azorica" | \
    awk -v name="${g}_azorica_Y" 'NR%2==1{print ">"name; next} {print}' | \
    (if [[ "$strand" == "-" ]]; then seqkit seq -t DNA -r -p; else cat; fi) \
    > "$GENE_DIR/${g}_azorica_Y.fasta"

done


##########################################
# AUTO-DETECTION genes_1_1
# (≥1 outgroup with pident ≥95)
##########################################

genes_1_1=()
for g in "${genes[@]}"
do
    persea_hit=$(awk '$3 >= 95' "$PERSEA_BLAST_DIR/${g}_Persea.tsv" | head -1)
    cinna_hit=$(awk '$3 >= 95' "$CINNA_BLAST_DIR/${g}_Cinna.tsv" | head -1)
    if [[ -n "$persea_hit" || -n "$cinna_hit" ]]; then
        genes_1_1+=("$g")
    fi
done

echo "Genes retained for phylogeny:"
printf '%s\n' "${genes_1_1[@]}"

################################################
# 8. Concat gene + MACSE trimnonHomologous
################################################


# Gene avec orthologues dans au moins 1 outgroups
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)


for g in "${genes_1_1[@]}"
do
    echo "MACSE $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    #################################
    # CONCAT + FORMAT + MACSE
    #################################

    cat $GENE_DIR/*.fasta | \
    awk '
    /^>/ {
        if (seq) print seq
        print
        seq=""
        next
    }
    {
        seq = seq $0
    }
    END {
        if (seq) print seq
    }
    ' | \
    grep -v '^$' > $GENE_DIR/${g}_all.fasta


    #################################
    # MACSE
    #################################

    macse \
    -prog trimNonHomologousFragments \
    -seq $GENE_DIR/${g}_all.fasta \
    -out_NT $GENE_DIR/${g}_MACSE_clean.fasta \
    -min_trim_in 60 \
    -min_trim_ext 30

done

##########################################
# 9. MACSE alignSequences
##########################################

# Gene avec orthologues 1:1 + trmming avec 2 outgroups restants
#genes_1_1=(g16702 g16717 g16721 g16742 g16746 g16765 g16766 g16767 g16789 g16791 g16793 g16804 g16813 g16814)

for g in "${genes_1_1[@]}"
do
    echo "ALIGNMENT $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    macse \
    -prog alignSequences \
    -seq "$GENE_DIR/${g}_MACSE_clean.fasta" \
    -out_NT "$GENE_DIR/${g}_aligned_NT.fasta" \
    -out_AA "$GENE_DIR/${g}_aligned_AA.fasta" \
    -max_refine_iter 10 

done

###########################################
# 10. GBLOCKS trimming
###########################################

for g in "${genes_1_1[@]}"
do
    echo "GBLOCKS $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    Gblocks "$GENE_DIR/${g}_aligned_NT.fasta" \
    -t=c 

done

###########################################
# 11. IQTREE 2 BEST MODEL 
###########################################


for g in "${genes_1_1[@]}"
do
    echo "TREE $g"

    GENE_DIR="$OUT_DIR/Gene_familly/$g"

    ALIGN="$GENE_DIR/${g}_aligned_NT.fasta-gb.fa"

    ########################
    # 1. MODEL SELECTION
    ########################

    iqtree3 -s "$ALIGN" -m MFP -B 1000 -T AUTO

    #MODEL=$(grep "Best-fit model" "$ALIGN.iqtree" | awk '{print $5}')

done

In [ ]:
#######
    ########################
    # 2. RAXML-NG TREE
    ########################

sed 's/!/N/g' /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_aligned_NT.fasta-gb.fa > tmp && mv tmp /Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_aligned_NT.fasta-gb.fa

raxml-ng \
    --all \
    --msa "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_aligned_NT.fasta-gb.fa" \
    --model GTR+G+FO \
    --prefix "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16814/g16814_RAxML" \
    --bs-trees autoMRE{2000} \
    --threads auto

raxml-ng \
    --all \
    --msa "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16717/g16717_aligned_NT.fasta-gb.fa" \
    --model GTR+G+FO \
    --prefix "/Users/cibio/Desktop/Laurus/SDPOP/Phylogeny_XY/phylo_genes/Gene_familly/g16717/g16717_RAxML" \
    --bs-trees autoMRE{2000} \
    --threads auto